# EasyRAG Query Rewrite And Multi Query

## Chapter position

This chapter starts once documents are already indexed. The focus is how a user question is prepared, recalled, filtered, fused, reranked, and finally turned into inspectable citations.

## Learning question

How do query rewrite and multi-query expansion widen recall without changing the outward query contract?

## Success criteria

- isolate rewrite and MQE behavior on the same question
- inspect how duplicate and overlapping query variants are handled
- compare retrieval results with and without expansion

## Source code anchors

- `easyrag.rag.retrieval.preprocess.QueryPreprocessor.prepare`
- `easyrag.rag.retrieval.preprocess.QueryPreparation`
- `easyrag.rag.types.QueryParam`


## Method principles

- `easyrag.rag.retrieval.preprocess.QueryPreprocessor.prepare`: This method builds the prepared query bundle. It normalizes the user input, optionally rewrites it, optionally expands it, and deduplicates the final retrieval query list.
- `easyrag.rag.retrieval.preprocess.QueryPreparation`: This dataclass is the output of query preprocessing. It keeps the original, normalized, rewritten, expanded, and final retrieval queries together for later inspection.
- `easyrag.rag.types.QueryParam`: This dataclass is the retrieval control surface. It does not run retrieval itself; it carries mode, top-k, rewrite, MQE, rerank, and filter policy into the pipeline.


## How the code fits together

The flow in this notebook is one question -> rewrite -> expanded queries -> retrieval variants. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.retrieval.preprocess import QueryPreprocessor

## What this cell is proving

We keep the question fixed and change only the preprocessing toggles. That makes the effect of rewrite and MQE easier to inspect.

In [ ]:
question = "How does query rewrite improve retrieval?"
preprocessor = QueryPreprocessor(_stub_query_model)
variants = {
    "baseline": QueryParam(rewrite_enabled=False, mqe_enabled=False),
    "rewrite_only": QueryParam(rewrite_enabled=True, mqe_enabled=False),
    "mqe_only": QueryParam(rewrite_enabled=False, mqe_enabled=True, mqe_variants=2),
    "rewrite_plus_mqe": QueryParam(
        rewrite_enabled=True, mqe_enabled=True, mqe_variants=2
    ),
}
preparations = {
    label: preprocessor.prepare(question, param) for label, param in variants.items()
}
for label, preparation in preparations.items():
    print(f"=== {label} ===")
    pprint(preparation)
    print()

## Why this output looks like this

The next cell runs retrieval with and without expansion so you can compare the final citations, not only the prepared-query list.

In [ ]:
rewrite_tmp = tempfile.TemporaryDirectory()
rewrite_rag = EasyRAG(
    working_dir=rewrite_tmp.name,
    workspace="rewrite-mqe-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(rewrite_rag.initialize_storages())
run_sync(
    rewrite_rag.ainsert(
        [
            "# Architecture\nQuery rewrite expands the candidate pool before retrieval.\n",
            "# FAQ\nMultiple query variants help coverage when the original question is underspecified.\n",
        ],
        ids=["doc::architecture", "doc::faq"],
        file_paths=["docs/architecture.md", "docs/faq.md"],
    )
)
base_result = run_sync(
    rewrite_rag.aquery(
        question,
        QueryParam(
            mode="hybrid", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
expanded_result = run_sync(
    rewrite_rag.aquery(
        question,
        QueryParam(
            mode="hybrid",
            rewrite_enabled=True,
            mqe_enabled=True,
            mqe_variants=2,
            chunk_top_k=3,
        ),
    )
)
_print_json(
    {
        "baseline": {
            "metadata": base_result.metadata,
            "citations": base_result.citations,
        },
        "expanded": {
            "metadata": expanded_result.metadata,
            "citations": expanded_result.citations,
        },
    }
)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The optional path uses the configured query model for rewrite and MQE when available. The notebook still focuses on the prepared-query artifact rather than on raw provider responses.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_preprocessor = QueryPreprocessor(EasyRAG().query_model_func)
    try:
        provider_preparation = provider_preprocessor.prepare(
            question, QueryParam(rewrite_enabled=True, mqe_enabled=True, mqe_variants=2)
        )
        pprint(provider_preparation)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- Keep query preparation, candidate generation, fusion, rerank, and hydration separate in your head. They fail in different ways.
- A weak citation list can come from filtering, ranking, or missing hydration, not only from poor recall.
- `QueryResult.metadata` is usually the fastest way to see what the pipeline actually did.

## Takeaway

Rewrite changes the phrasing of the core question. MQE adds extra variants around that rewritten question. EasyRAG keeps both steps visible in `QueryPreparation` so you can tell whether better retrieval came from stronger phrasing, broader coverage, or both.

In [ ]:
run_sync(rewrite_rag.finalize_storages())
rewrite_tmp.cleanup()
print("Cleaned up the rewrite-and-MQE workspace.")

## Where to go next

- Continue with [04_03_naive_retrieval_basics.ipynb](04_03_naive_retrieval_basics.ipynb) to see the simplest ranking path once the queries are prepared.
- Read [principles/10-query-preprocessing.md](../../docs/principles/10-query-preprocessing.md) for the conceptual view of rewrite and MQE.
- Revisit [04_01_query_normalization_and_preprocessing.ipynb](04_01_query_normalization_and_preprocessing.ipynb) if you want the broader preprocessing view again.